# Phase 19 — Ensemble / proposed model vs a comparator fixed BEFORE test-set evaluation
**Resolves:** R2-6 (selection bias from choosing the "best base learner" on the test set; exact paired-bootstrap algorithm),
R2-11 (uniform RF vs selected), and the Discussion claim that "every CI included zero" (which the original notebook 07
output contradicts for Arab depression).

**Comparator rule (fixed in advance, training data only):** the single base learner with the highest *training*
out-of-fold AUC. Candidates compared to it, all on the SAME test rows with the SAME bootstrap resamples (paired):
proposed RF, one-SE selected candidate, NNLS, equal-weight, AUC-optimised weights, logistic stacking, greedy selection.
All ensemble weights are fitted on training OOF predictions only. Holm adjustment across the family within each dataset.
**Outputs:** `S2_ensemble_vs_fixed_comparator.csv`, `S2_test_auc_all_strategies.csv`

In [1]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))
from revision_utils import *

ENV = get_env()          # paths + config (notebooks must be run from the notebooks/ folder)
SEED = ENV["SEED"]       # 42
B = ENV["N_BOOT"]        # 2000 bootstrap resamples (config.yaml)
print("Project root:", ENV["ROOT"], "| outputs ->", ENV["REV"])

Project root: C:\maternal-mental-health-risk | outputs -> C:\maternal-mental-health-risk\reports/tables\revision


In [2]:
from sklearn.metrics import roc_auc_score
rows, allrows = [], []
STRATS = ["nnls", "equal_weight", "auc_optimized", "logistic_stacking", "greedy_selection"]
for tag in DATASETS:
    S = load_split(tag, ENV); oof, te = S["oof"], S["pred_test"]; y = te["y_true"].values
    oof_auc = {m: roc_auc_score(oof["y_true"], oof[m]) for m in MODEL_NAMES}
    comp_name = max(oof_auc, key=oof_auc.get)               # fixed from TRAINING OOF only
    p_comp = te[comp_name].values
    comp_tbl = pd.read_csv(os.path.join(ENV["MODELS"], "ensemble_final", tag, "strategy_comparison.csv"))
    sel_name, *_ = one_se_select(comp_tbl)
    cands = {"proposed RF": "single_rf", f"1-SE selected ({sel_name})": sel_name}
    cands.update({s: s for s in STRATS})
    tmp = []
    for label, name in cands.items():
        p = strategy_proba(name, oof, te, seed=SEED)
        est, lo, hi = bootstrap_ci_full(y, p, B=B, seed=SEED)
        allrows.append(dict(dataset=tag, candidate=label, test_auc=est, ci_lower=lo, ci_upper=hi))
        same = (name == f"single_{comp_name}")
        if same:
            tmp.append(dict(dataset=tag, comparator=comp_name, comparator_oof_auc=oof_auc[comp_name], candidate=label,
                            candidate_test_auc=est, delta=0.0, ci_lower=0.0, ci_upper=0.0, p_two_sided=1.0, identical_to_comparator=True))
            continue
        dl = paired_bootstrap_delta(y, p, p_comp, B=B, seed=SEED)
        tmp.append(dict(dataset=tag, comparator=comp_name, comparator_oof_auc=oof_auc[comp_name], candidate=label,
                        candidate_test_auc=est, delta=dl["delta"], ci_lower=dl["ci_lower"], ci_upper=dl["ci_upper"],
                        p_two_sided=dl["p_two_sided"], identical_to_comparator=False))
    t = pd.DataFrame(tmp)
    mask = ~t.identical_to_comparator
    t.loc[mask, "p_holm"] = holm_adjust(t.loc[mask, "p_two_sided"].values)
    t["ci_excludes_zero"] = (t.ci_lower > 0) | (t.ci_upper < 0)
    rows.append(t)
res = pd.concat(rows, ignore_index=True)
save_table(res, ENV, "S2_ensemble_vs_fixed_comparator.csv")
save_table(pd.DataFrame(allrows), ENV, "S2_test_auc_all_strategies.csv")
print(res.round(4).to_string())

Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_ensemble_vs_fixed_comparator.csv  (28 rows)
Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_test_auc_all_strategies.csv  (28 rows)
             dataset comparator  comparator_oof_auc                   candidate  candidate_test_auc   delta  ci_lower  ci_upper  p_two_sided  identical_to_comparator  p_holm  ci_excludes_zero
0           pakistan        svm              0.5418                 proposed RF              0.5571 -0.0050   -0.0128    0.0030       0.2210                    False   0.884             False
1           pakistan        svm              0.5418  1-SE selected (single_svm)              0.5621  0.0000    0.0000    0.0000       1.0000                     True     NaN             False
2           pakistan        svm              0.5418                        nnls              0.5602 -0.0019   -0.0121    0.0088       0.7160                    False   1.000             False
3           pakista

**Algorithm to quote in Methods (exact):** for b = 1..2000, draw n test indices with replacement (one index vector per
iteration, shared by both models); resamples containing a single class are skipped; compute AUC_A and AUC_B on that index vector;
delta_b = AUC_A - AUC_B. Report the full-sample delta, the 2.5th/97.5th percentiles of delta_b, and a two-sided bootstrap p-value
2*min(P(delta_b<=0), P(delta_b>=0)); Holm-adjust within dataset. The comparator is the base learner with the highest
training out-of-fold AUC (fixed before any test-set evaluation).